# The Op as an assembly

> A Gate is a decision point in an Op where the results of one or more
> Guards determine what happens next. Guards check. Gates decide.
>
> — [@oliphant2026, §5.2]

The section 5.6 manifest declares four such decisions:

> gates:
>   - if: confidence < 0.80
>     then: human_review_required
>   - if: consensus_disagreement > 0.25
>     then: expert_review_required
>   - if: sensitive_data_detected
>     then: stop_and_escalate
>   - if: vendor_risk == high
>     then: human_approval_required
>
> — [@oliphant2026, §5.6]

The model renders the Op as an assembly of abstract and concrete parts
wired along ports: four oracle components feed a policy engine across
reading seams. (An *oracle* is a process that answers a question: an
identified service the Op queries for a value it does not compute
itself, with each exchange cited — service, request payload, response
code, response. The term is the adjudicator's, introduced by the GAP-05
ruling and defined in the judgment record; the paper does not use it.
The usage follows the oracle machine of computability theory
[@turing1939]: a machine that consults an external source for answers
it does not compute.)
Raised obligations cross to a counterparty outside the modeled
boundary; clearance crosses to the anomaly-summary cog, whose
output the assembly's boundary exposes. On who that counterparty is,
the model follows the source exactly: review and approval are named
human by the manifest's own tokens, expert review is named human by the
paper's prose ("routes sampled or high-risk outputs to human review",
[@oliphant2026, §5.1]; "Human or known-expert judgment",
[@oliphant2026, §5.5]), and attribute names keep the manifest's words.
Only the escalation target is left unspecified: there the discharging
actor is modeled as another system acting on the Op from outside what
is modeled, with a party assumed accountable for its behavior. That
accountability assumption is the adjudicator's interpretation, not the
paper's, and is logged as GAP-09 in the open-questions chapter. One toolchain then
checks three levels at once:

- **component**: each `if/then` line as an `implies` constraint on the
  policy engine (the verbatim gates, with the numerical parameters
  factored out as single-point definitions);
- **wiring**: seam integrity (the value the engine ruled on is the value
  the oracle returned), plus structural seam conformance checked over
  the RDF conversion;
- **system**: properties only the composed Op has. Every raised
  obligation is discharged across components; a stopped Op emits
  nothing; and the aggregate outcome (executed vs noOp) must cohere
  with what was emitted. Navigating the rules and executing are
  different facts, and the model says which happened.

The internal wiring topology is not drawn in the paper; making it
explicit and checkable is a contribution of the computational form, and
is logged as such rather than passed off as the paper's own (GAP-08 in
the open-questions chapter).

## The factored parameters

The manifest's numbers appear in the model exactly once each, as named
parameters. Every gate references the parameter, never the literal, so
a whitepaper can repeat a threshold but this implementation cannot:
there is one point of definition, and the Track records the values in
force per run. The table is parsed from the model text itself, nothing
retyped:

In [1]:
import sys; sys.path[:0] = [".", ".."]  # the repo root, from either cwd
import exhibits
exhibits.show_policy_parameters()

Parameter,Value,Type,Manifest rule it backs
confidenceThreshold,0.80,Real,GATE-01: - if: confidence < 0.80
consensusDisagreementThreshold,0.25,Real,GATE-02: - if: consensus_disagreement > 0.25
vendorRiskTrigger,RiskLevel::high,RiskLevel,GATE-04: - if: vendor_risk == high


For readers who want the substrate itself, this is the defining block
as written. If you read SysML the way you read Python, a `part def` is
roughly a frozen dataclass: three fields with defaults, where the
defaults are the manifest's literals. The doc comment carries the
rationale and travels with the definition wherever the model goes.

In [2]:
exhibits.show_policy_parameters_source()

part def ValidationStrategyParameters {
    doc /* The numerical parameters of the policy, factored out as
       single-point definitions the gates reference — what an
       implementation can do that a whitepaper cannot. The defaults
       are the manifest's literals, verbatim; changing a threshold
       is a change HERE, nowhere else, and the Track records the
       values in force per run (WP section 5.3: Track contents
       include "the model versions and configuration parameters"). */
    attribute confidenceThreshold : ScalarValues::Real = 0.80;
    attribute consensusDisagreementThreshold : ScalarValues::Real = 0.25;
    attribute vendorRiskTrigger : RiskLevel = RiskLevel::high;
}


## The gates as constraints

Each gate keeps its manifest lines verbatim in its documentation and
states the same rule as a machine-evaluable implication. Reading the
`As evaluated` column: `pe` is the policy engine the check is applied
to, and `implies` is the material conditional, so the row for GATE-01
reads "if the confidence reading is below the threshold, then the
human-review obligation must have been raised". A run where the left
side is true and the right side is false is not a warning; it fails to
satisfy. All four, parsed from the model:

In [3]:
exhibits.show_gate_checks()

Gate,"Manifest condition, verbatim",Manifest consequence,As evaluated
GATE-01,- if: confidence < 0.80,then: human_review_required,pe.confidence < pe.policy.confidenceThreshold implies pe.humanReviewRequired
GATE-02,- if: consensus_disagreement > 0.25,then: expert_review_required,pe.consensusDisagreement > pe.policy.consensusDisagreementThreshold implies pe.expertReviewRequired
GATE-03,- if: sensitive_data_detected,then: stop_and_escalate,pe.sensitiveDataDetected implies pe.stopAndEscalate
GATE-04,- if: vendor_risk == high,then: human_approval_required,pe.vendorRisk == pe.policy.vendorRiskTrigger implies pe.humanApprovalRequired


And one gate in source form, to show how little distance there is
between the manifest and the evaluated rule: the doc quotes the
manifest lines, and the constraint restates them over the factored
parameter a few lines below in the same block:

In [4]:
exhibits.show_gate_source()

requirement def <'GATE-01'> ConfidenceGate {
    doc /* - if: confidence < 0.80
            then: human_review_required
       Source: WP section 5.6 gates block, verbatim. */
    subject pe : PolicyEngine;
    require constraint { pe.confidence < pe.policy.confidenceThreshold implies pe.humanReviewRequired }
}


## The seams

Each dataflow between components is one declared seam: a typed
connection from a supplier port to a consumer port inside the assembly.
One interface type per kind of flow (readings, obligations, clearance),
so a port carrying readings cannot be wired where a summary belongs;
that refusal is demonstrated below.

In [5]:
exhibits.show_seams()

Seam,Interface type,From (supplier port),To (consumer port)
confidenceSeam,ReadingSeam,confidenceOracle.reading,policyEngine.fromConfidence
consensusSeam,ReadingSeam,consensusOracle.reading,policyEngine.fromConsensus
scannerSeam,ReadingSeam,scannerOracle.reading,policyEngine.fromScanner
riskSeam,ReadingSeam,riskOracle.reading,policyEngine.fromRisk
obligationSeam,ObligationSeam,policyEngine.toCounterparty,counterparty.fromEngine
clearanceSeam,ClearanceSeam,policyEngine.toCog,summaryCog.clearance


## The whole specification evaluates

Strict validation, then the satisfy sweep: every component, wiring, and
system check evaluated together against the committed run
configurations. The exit code is the finding.

In [6]:
exhibits.validate_and_satisfy()

validate -strict exit code: 0
✓ package VendorFraudReview
✓ satisfy cleanInterface01 holds
✓ satisfy cleanType01 holds
✓ satisfy cleanGate01 holds
✓ satisfy cleanGate02 holds
✓ satisfy cleanGate03 holds
✓ satisfy cleanGate04 holds
✓ satisfy cleanWire01 holds
✓ satisfy cleanWire02 holds
✓ satisfy cleanWire03 holds
✓ satisfy cleanWire04 holds
✓ satisfy cleanSystem01 holds
✓ satisfy cleanSystem02 holds
✓ satisfy cleanSystem03 holds
✓ satisfy escalatedInterface01 holds
✓ satisfy escalatedType01 holds
✓ satisfy escalatedGate01 holds
✓ satisfy escalatedGate02 holds
✓ satisfy escalatedGate03 holds
✓ satisfy escalatedGate04 holds
✓ satisfy escalatedWire01 holds
✓ satisfy escalatedWire02 holds
✓ satisfy escalatedWire03 holds
✓ satisfy escalatedWire04 holds
✓ satisfy escalatedSystem01 holds
✓ satisfy escalatedSystem02 holds
✓ satisfy escalatedSystem03 holds
satisfy exit code: 0


## The specification can say no

A specification that cannot fail is a description. The counterexample
file holds four runs that must be refused, one per check level: a gate
fired but no obligation was raised (component); the obligation was
raised and never discharged (system); the Op stopped but transmitted
anyway (aggregate); the consensus oracle returned 1.4, outside the
metric class's declared codomain of [0, 1] (type, TYPE-01). The same
evaluation names each violated implication:

In [7]:
exhibits.refuse_counterexample()

✓ package VendorFraudReview
✗ satisfy unattendedGate01 fails
  Required condition evaluated to false: pe.confidence < pe.policy.confidenceThreshold implies pe.humanReviewRequired
✗ satisfy undischargedSystem01 fails
  Required condition evaluated to false: op.policyEngine.humanReviewRequired implies op.counterparty.humanReviewPerformed
✗ satisfy emittedSystem02 fails
  Required condition evaluated to false: op.policyEngine.stopAndEscalate implies not op.summaryCog.outputEmitted
✗ satisfy outOfRangeType01 fails
  Required condition evaluated to false: pe.consensusDisagreement <= 1.0
satisfy exit code: 1


## The seams themselves are checked

Strict validation accepts a seam whose ends do not match its interface
definition (probed against the pinned binary), so the structural wiring
rules live in code over the RDF conversion: the model holds the wiring,
the rules hold the wiring discipline. The same rule that passes the
committed assembly refuses a miswired counterexample in which an
oracle's reading port is wired into a summary input:

In [8]:
exhibits.check_wiring()

committed assembly (6 seams): wiring rules: OK
miswired counterexample, validate -strict exit code: 0 (accepted)
miswired counterexample: wiring rules: REFUSED
  confidenceSeam: end 'summaryCog.summaryIn' is urn:sysmlv2:element:VendorFraudReviewMiswired__SummaryWrite but the seam declares urn:sysmlv2:element:VendorFraudReviewMiswired__ReadingWrite


## The stop is a state

> If a Privacy Guard detects sensitive information, the Op stops before
> external transmission.
>
> — [@oliphant2026, §5.2]

The assembly exhibits a lifecycle (running to stopped, or running to
completed) that runs under the pinned executor: a driver component
delivers a signal across a connected seam to the assembly's own port,
and the trace shows the transition. The stop the paper describes is an
actual state the Op enters, not a label on a diagram:

In [9]:
exhibits.show_lifecycle()

stoppedContext
  [trace] enter: firing (entry action)
  [trace] enter: running
  [trace] enter: stopped
  [trace] transition: running -> stopped (event: accept Escalation)
  [trace] enter: spent
  [trace] transition: firing -> spent

completedContext
  [trace] enter: firing (entry action)
  [trace] enter: running
  [trace] enter: completed
  [trace] transition: running -> completed (event: accept Proceed)
  [trace] enter: spent
  [trace] transition: firing -> spent



## Where every element comes from

Code is not objective; every named element of this model owes its
existence to something, and says what. The map below renders
`model/trace.ttl`: each element is grounded in the pinned source
(locators, with quotes verified verbatim inside the cited section by
the test suite), derived from a recorded judgment in the open-questions
chapter's record, or declared toolchain scaffolding with a rationale. The map is
machine-checked complete in both directions: an element without a trace
entry, or an entry without a matching element, fails the build.

In [10]:
exhibits.show_element_trace()

Element,Kind,Provenance
completedContext,context,scaffold: trace context: drives the lifecycle to completed under -trace -instantiate
stoppedContext,context,scaffold: trace context: drives the lifecycle to stopped under -trace -instantiate
AggregateOutcome,enum def,grounded in §5.2; derived from ruling-gap06
ConfidenceLevel,enum def,"derived from ruling-gap03, note-categories"
DenominatorBasis,enum def,derived from ruling-gap11
NullPolicy,enum def,derived from ruling-gap04-class-split
RiskLevel,enum def,"derived from ruling-gap01, note-categories"
ClearanceSeam,interface def,derived from ruling-gap08
ObligationSeam,interface def,"derived from ruling-gap08, ruling-gap09"
ProceedWire,interface def,scaffold: driver-to-assembly signal seam for exercising the lifecycle under trace


traced elements: 69 — 18 grounded in the pinned source, 40 from recorded judgments alone, 11 declared scaffolding


## Conversion to RDF

The model converts to Turtle [@rdf2014] deterministically, and the
checks survive the conversion, so the same specification is queryable
alongside the vocabulary and the Track:

In [11]:
exhibits.check_conversion()

converted triples : 10881
byte-stable       : True
checks present    : 13 of 13 (INTERFACE-01, TYPE-01, GATE-01, GATE-02, GATE-03, GATE-04, WIRE-01, WIRE-02, WIRE-03, WIRE-04, SYSTEM-01, SYSTEM-02, SYSTEM-03)
